# Texture generator: the Python workflow

Generate procedural textures entirely through the importable `texture_generators`
package. This notebook covers discovering materials, Pillow images, NumPy arrays,
all variants, repeatable seeds, material controls, paper measurements and contact
sheets. It finishes by exporting a [Markdown results gallery](results.md) and a
[recipe manifest](manifest.json).

**Python 3.14 or later is required.** From the repository root:

```bash
uv sync --group examples
uv run --group examples jupyter lab examples/texture_generator_workflow.ipynb
```

Select the Python 3 kernel, then **Restart Kernel and Run All Cells**. Alternatively,
run it without a browser:

```bash
uv run --group examples jupyter execute examples/texture_generator_workflow.ipynb
```

For a separate project, install `texture-generator==0.3.0` and `jupyterlab`, then
open a copy of this notebook. The notebook imports the installed package; no
`sys.path` edits or command-line subprocesses are used for generation.

PNG files live in `images/` beside this notebook. Image previews use Markdown
links, so the notebook contains no embedded image data. A floating-point array
is saved separately in `data/`. Running all cells again overwrites the named
example files, `results.md` and `manifest.json`. Start from a fresh kernel when
regenerating the complete set. Rendering can take a few minutes.

## 1. Import the package and choose an output directory

The distribution is `texture-generator`; its import is `texture_generators`.
Record the environment because exact pixel reproduction also depends on the
package, dependency versions and platform. With the repository lockfile, the
versions below are installed together.

In [1]:
import hashlib
import json
import platform
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display
from PIL import Image

from texture_generators import (
    VARIANTS_BY_MATERIAL,
    all_pairs,
    generate,
    generate_array,
    resolve_variant,
    sample_sheet,
    to_image,
    variants,
)

environment = {
    "python": sys.version.split()[0],
    "platform": sys.platform,
    "architecture": platform.machine(),
    **{name: version(name) for name in ("texture-generator", "numpy", "pillow")},
}
print(json.dumps(environment, indent=2))

# Jupyter normally starts beside the notebook; also support the repository root.
example_dir = Path.cwd()
if (example_dir / "examples" / "texture_generator_workflow.ipynb").is_file():
    example_dir = example_dir / "examples"
image_dir = example_dir / "images"
data_dir = example_dir / "data"
image_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)
records = []

{
  "python": "3.14.7",
  "platform": "darwin",
  "architecture": "arm64",
  "texture-generator": "0.3.0",
  "numpy": "2.5.2",
  "pillow": "12.3.0"
}


This small notebook helper saves an image, records how it was produced and displays
an external file link. It does not change the generated pixels. The final cell uses
these records to build the complete gallery.

In [2]:
def save_png(
    image: Image.Image,
    filename: str,
    title: str,
    section: str,
    recipe: dict,
) -> None:
    """Save a PNG and register it for the external results gallery."""
    path = image_dir / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    image.save(path, format="PNG")
    relative_path = path.relative_to(example_dir).as_posix()
    records.append(
        {
            "path": relative_path,
            "title": title,
            "section": section,
            "width": image.width,
            "height": image.height,
            "mode": image.mode,
            "recipe": recipe,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
    )
    display(Markdown(f"**{title}**\n\n![{title}]({relative_path})"))

## 2. Discover materials and variants

Use the public registry rather than hard-coding a list for a batch. `variants()`
returns the choices for one material; `all_pairs()` gives every material/variant
combination in registry order.

In [3]:
for material, choices in VARIANTS_BY_MATERIAL.items():
    print(f"{material}: {', '.join(choices)}")
print("Wood choices:", variants("wood"))
pairs = all_pairs()
print(f"{len(pairs)} material/variant combinations")

metal: brushed, radial, polished, heat_tinted, oil_film, anodised_titanium, engine_turned
plastic: glossy, matte, textured
wood: board, planks
paper: white, kraft, recycled, newsprint, laid, coated
Wood choices: ['board', 'planks']
18 material/variant combinations


## 3. Generate, save and reopen a Pillow image

An integer size makes a square. A pair means **(width, height)**, and both dimensions
must be at least 2 pixels. `generate()` returns an RGB Pillow image. This is the
README quickstart with an explicit variant and seed.

In [4]:
wood_args = {"size": (640, 480), "seed": 42, "variant": "board"}
wood = generate("wood", **wood_args)
assert wood.mode == "RGB" and wood.size == (640, 480)
save_png(
    wood,
    "wood-board.png",
    "Wood / board — Pillow quickstart",
    "Pillow image",
    {"function": "generate", "material": "wood", **wood_args},
)
with Image.open(image_dir / "wood-board.png") as reopened:
    assert reopened.mode == wood.mode and reopened.size == wood.size
    assert reopened.tobytes() == wood.tobytes()
print("Saved and reopened PNG:", wood.mode, wood.size)

**Wood / board — Pillow quickstart**

![Wood / board — Pillow quickstart](images/wood-board.png)

Saved and reopened PNG: RGB (640, 480)


## 4. Work with floating-point pixels

`generate_array()` returns a NumPy `float32` array shaped **(height, width, 3)**,
with RGB values in `[0, 1]`. Keep the array for numerical work; `to_image()` converts
it to an 8-bit RGB Pillow image for PNG export. It produces the same pixels as
`generate()` called with the same arguments.

The `.npy` file preserves the floating-point values; the PNG is their display
representation. This example checks both round trips.

In [5]:
array_args = {"size": 256, "seed": 7, "variant": "brushed"}
pixels = generate_array("metal", **array_args)
assert pixels.shape == (256, 256, 3) and pixels.dtype == np.float32
assert np.isfinite(pixels).all() and 0 <= pixels.min() <= pixels.max() <= 1
np.save(data_dir / "metal-brushed.npy", pixels)
assert np.array_equal(np.load(data_dir / "metal-brushed.npy"), pixels)
array_image = to_image(pixels)
assert array_image.tobytes() == generate("metal", **array_args).tobytes()
print("Array:", pixels.shape, pixels.dtype)
print("Range:", float(pixels.min()), float(pixels.max()))
save_png(
    array_image,
    "metal-array.png",
    "Metal / brushed — NumPy array as a PNG",
    "NumPy array",
    {
        "function": "generate_array",
        "material": "metal",
        **array_args,
        "conversion": "to_image",
        "array_path": "data/metal-brushed.npy",
    },
)

Array: (256, 256, 3) float32
Range: 0.2375497817993164 0.8513011932373047


**Metal / brushed — NumPy array as a PNG**

![Metal / brushed — NumPy array as a PNG](images/metal-array.png)

## 5. Export every variant in one batch

Each tile below uses an explicit variant, seed **42** and size **384 × 384**.
These are the same files used by the README gallery. Changing the size changes
the render; it is not simply a resized crop of another image.

In [6]:
for material, variant in pairs:
    args = {"size": 384, "seed": 42, "variant": variant}
    tile = generate(material, **args)
    save_png(
        tile,
        f"variants/{material}-{variant}.png",
        f"{material} / {variant}",
        f"All variants — {material}",
        {"function": "generate", "material": material, **args},
    )
print(f"Exported {len(pairs)} variants.")

**metal / brushed**

![metal / brushed](images/variants/metal-brushed.png)

**metal / radial**

![metal / radial](images/variants/metal-radial.png)

**metal / polished**

![metal / polished](images/variants/metal-polished.png)

**metal / heat_tinted**

![metal / heat_tinted](images/variants/metal-heat_tinted.png)

**metal / oil_film**

![metal / oil_film](images/variants/metal-oil_film.png)

...

**paper / recycled**

![paper / recycled](images/variants/paper-recycled.png)

**paper / newsprint**

![paper / newsprint](images/variants/paper-newsprint.png)

**paper / laid**

![paper / laid](images/variants/paper-laid.png)

**paper / coated**

![paper / coated](images/variants/paper-coated.png)

Exported 18 variants.


## 6. Reproduce a render and vary the seed

The same arguments produce identical pixels in the same environment. A different
seed changes the material realisation. Store the material, variant, seed, size
and any overrides together rather than recording only the seed.

In [7]:
first = generate("plastic", size=384, seed=12, variant="matte")
repeat = generate("plastic", size=384, seed=12, variant="matte")
second = generate("plastic", size=384, seed=13, variant="matte")
assert first.tobytes() == repeat.tobytes()
assert first.tobytes() != second.tobytes()
for seed, texture in ((12, first), (13, second)):
    save_png(
        texture,
        f"plastic-seed-{seed}.png",
        f"Plastic / matte — seed {seed}",
        "Reproducible seeds",
        {
            "function": "generate",
            "material": "plastic",
            "size": 384,
            "seed": seed,
            "variant": "matte",
        },
    )
print("Same arguments: identical pixels. Different seed: different pixels.")

**Plastic / matte — seed 12**

![Plastic / matte — seed 12](images/plastic-seed-12.png)

**Plastic / matte — seed 13**

![Plastic / matte — seed 13](images/plastic-seed-13.png)

Same arguments: identical pixels. Different seed: different pixels.


### Let the seed choose the variant

With `variant=None` (or omitted), the first random draw chooses a variant.
`resolve_variant()` tells you which one a concrete seed will choose.
To reproduce this render, **keep the variant omitted**. Passing the resolved name
explicitly skips that first draw and changes the later random draws. A missing
seed uses fresh entropy and cannot predict a separate unseeded render.

In [8]:
automatic_seed = 23
chosen = resolve_variant("metal", seed=automatic_seed)
automatic = generate("metal", size=384, seed=automatic_seed)
assert (
    automatic.tobytes()
    == generate("metal", size=384, seed=automatic_seed, variant=None).tobytes()
)
print("Seed-selected variant:", chosen)
save_png(
    automatic,
    "metal-automatic.png",
    f"Metal / {chosen} — seed-selected variant",
    "Automatic variant selection",
    {
        "function": "generate",
        "material": "metal",
        "size": 384,
        "seed": automatic_seed,
        "variant": None,
        "resolved_variant": chosen,
    },
)

Seed-selected variant: brushed


**Metal / brushed — seed-selected variant**

![Metal / brushed — seed-selected variant](images/metal-automatic.png)

## 7. Control the material

Keyword arguments after `size`, `seed` and `variant` are material-specific.
Here a fixed walnut board is rendered unfinished and oiled, then brushed metal
gets an oil film. The [parameter reference](https://github.com/nmpowell/texture-generator/blob/main/docs/reference.md#material-parameters)
lists supported controls. Unknown extra keywords are currently ignored, so use
those documented names and the appropriate material.

In [9]:
for finish in ("none", "oil"):
    args = {
        "size": 384,
        "seed": 17,
        "variant": "board",
        "species": "walnut",
        "finish": finish,
        "cut": "quartersawn",
        "figure": "plain",
        "mm_across": 180,
        "knots": 0,
        "sapwood": 0,
    }
    controlled = generate("wood", **args)
    save_png(
        controlled,
        f"walnut-{finish}.png",
        f"Walnut / quartersawn — finish {finish}",
        "Material controls",
        {"function": "generate", "material": "wood", **args},
    )
film_args = {
    "size": 384,
    "seed": 17,
    "variant": "brushed",
    "film": {"system": "oil", "nm": (120, 800), "field": "spill"},
}
save_png(
    generate("metal", **film_args),
    "metal-custom-film.png",
    "Brushed metal / oil film",
    "Material controls",
    {"function": "generate", "material": "metal", **film_args},
)

**Walnut / quartersawn — finish none**

![Walnut / quartersawn — finish none](images/walnut-none.png)

**Walnut / quartersawn — finish oil**

![Walnut / quartersawn — finish oil](images/walnut-oil.png)

**Brushed metal / oil film**

![Brushed metal / oil film](images/metal-custom-film.png)

## 8. Inspect paper measurements

Paper can also fill an `out` dictionary with two scalar fields: `mass` (fibre
coverage) and `formation` (normalised formation). The returned RGB array still
uses the usual contract. This is optional diagnostic data, not a separate set
of PBR texture maps.

Set `creases=0.0` for seamless paper. The field previews below are each scaled
from their own minimum to maximum for display; **their greyscale values are not
physical units**. Use the original arrays for measurements. Ordinary material
outputs are shaded RGB images; the public API does not export general normal,
roughness or albedo maps.

In [10]:
fields = {}
paper_args = {
    "size": 384,
    "seed": 7,
    "variant": "laid",
    "mm_across": 30,
    "creases": 0.0,
}
paper_pixels = generate_array("paper", **paper_args, out=fields)
save_png(
    to_image(paper_pixels),
    "paper-measured.png",
    "Paper / laid — measurement example",
    "Paper measurements",
    {
        "function": "generate_array",
        "material": "paper",
        **paper_args,
        "conversion": "to_image",
        "measurements": ["mass", "formation"],
    },
)
for name in ("mass", "formation"):
    field = fields[name]
    assert field.shape == paper_pixels.shape[:2] and np.isfinite(field).all()
    print(f"{name}: shape={field.shape}, mean={float(field.mean()):.4f}")
    scaled = (field - field.min()) / np.ptp(field)
    preview = to_image(np.repeat(scaled[..., None], 3, axis=2))
    save_png(
        preview,
        f"paper-{name}.png",
        f"Paper / {name} — normalised greyscale preview",
        "Paper measurements",
        {
            "derived_from": "images/paper-measured.png",
            "field": name,
            "conversion": "min-max normalisation, greyscale to RGB",
        },
    )

**Paper / laid — measurement example**

![Paper / laid — measurement example](images/paper-measured.png)

mass: shape=(384, 384), mean=11.2406


**Paper / mass — normalised greyscale preview**

![Paper / mass — normalised greyscale preview](images/paper-mass.png)

formation: shape=(384, 384), mean=-0.0000


**Paper / formation — normalised greyscale preview**

![Paper / formation — normalised greyscale preview](images/paper-formation.png)

## 9. Make a labelled contact sheet

`sample_sheet()` includes every material/variant pair in a single Pillow image.
`size` is the size of each tile; `columns` controls the layout. The master seed
draws a separate seed for each tile, so these tiles differ from the seed-42
batch above. Labels abbreviate tile seeds; keep the master seed and call arguments
to regenerate the whole sheet.

In [11]:
sheet_args = {"size": 192, "seed": 7, "columns": 4}
sheet = sample_sheet(**sheet_args)
save_png(
    sheet,
    "contact-sheet.png",
    "All materials — labelled contact sheet",
    "Contact sheet",
    {"function": "sample_sheet", **sheet_args},
)
print("Contact sheet:", sheet.mode, sheet.size)

**All materials — labelled contact sheet**

![All materials — labelled contact sheet](images/contact-sheet.png)

Contact sheet: RGB (798, 1066)


## 10. Handle invalid input

Unknown materials, unknown variants and dimensions below 2 pixels raise
`ValueError`. Validate application input or handle the error at your boundary.

In [12]:
try:
    generate("wood", size=1, seed=42, variant="board")
except ValueError as error:
    print(f"Expected validation error: {error}")

Expected validation error: size must be at least 2x2, got 1x1


## 11. Write the complete results gallery

The Markdown document references every PNG saved in this run. The JSON manifest
records dimensions, recipes, environment versions and PNG checksums. Recipe
entries include descriptive fields such as `function` and `conversion`; pass
only the documented generator arguments back to the API.

Run all cells when changing parameters so the PNGs, notebook outputs, gallery
and manifest agree. Old files from recipes you remove are not automatically deleted.

In [13]:
paths = [record["path"] for record in records]
assert len(paths) == len(set(paths)), "Use a different filename for each result."
manifest = {"environment": environment, "images": records}
(example_dir / "manifest.json").write_text(
    json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
)
lines = [
    "# Texture generator: notebook results",
    "",
    "Generated by [the Python workflow notebook](texture_generator_workflow.ipynb).",
    "Run all cells to regenerate these PNG files and this document. Images are",
    "stored separately in `images/`; no image data is embedded in the notebook.",
    "",
    "Every image from the notebook is shown below. The [recipe manifest](manifest.json)",
    "records the exact arguments and PNG checksums. Exact reproduction also depends",
    "on the dependency versions and platform. The saved floating-point array is",
    "[metal-brushed.npy](data/metal-brushed.npy).",
    "",
    "## Environment",
    "",
    "```json",
    json.dumps(environment, indent=2),
    "```",
    "",
    "All variant-gallery tiles use seed 42 and size 384 x 384. The contact sheet",
    "uses its own master seed and draws individual tile seeds.",
    "",
    "Paper measurement previews use min-max greyscale normalisation for display;",
    "their displayed brightness is not a physical unit.",
    "",
]
previous_section = None
for record in records:
    if record["section"] != previous_section:
        lines.extend([f"## {record['section']}", ""])
        previous_section = record["section"]
    lines.extend(
        [
            f"### {record['title']}",
            "",
            f"{record['width']} x {record['height']} · {record['mode']}",
            "",
            f"![{record['title']}]({record['path']})",
            "",
        ]
    )
(example_dir / "results.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Saved {len(records)} PNGs, the NumPy array, manifest.json and results.md.")
display(Markdown("[Open the complete results gallery](results.md)"))

Saved 30 PNGs, the NumPy array, manifest.json and results.md.


[Open the complete results gallery](results.md)